# einops-repeat-broadcast — worked example 2: Broadcast a per-channel scale over an image batch

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `einops-repeat-broadcast`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`einops.repeat` can introduce several new axes at once. Naming the new axes and giving each a size produces a stride-zero view that lines up with a larger feature-map tensor for an elementwise op. This is the per-channel affine pattern from normalization layers: one scalar per channel, broadcast across batch, height, and width.

## Worked solution

We hold a per-channel `scale` vector of shape `(C,)` and want to multiply it into an activation tensor of shape `(N, C, H, W)`.

1. Pull the sizes we need from the activation tensor: `N, C, H, W = x.shape`. The channel count of `scale` must match `C`.
2. Call `repeat(scale, 'c -> n c h w', n=N, h=H, w=W)`. The single input axis `c` stays in the channel slot; the three brand-new axes `n`, `h`, `w` are inserted around it, each with stride 0.
3. The result has shape `(N, C, H, W)` but only `C` real values in memory — every batch element, row, and column re-reads the same channel scalar.
4. We return the broadcast view itself (the drill is about the view, not the multiply), then demonstrate it multiplies cleanly against `x`.

In [ ]:
import torch as t
import einops
from einops import repeat

t.manual_seed(1)
x = t.randn(2, 3, 4, 4)
scale = t.randn(3)

def broadcast_channel_scale(scale, x):
    N, C, H, W = x.shape
    return repeat(scale, 'c -> n c h w', n=N, h=H, w=W)

scale_b = broadcast_channel_scale(scale, x)
scaled = x * scale_b
print(scale_b.shape, scaled.shape)
print('channel 0 constant across H,W:', bool((scale_b[0, 0] == scale[0]).all()))